In [ ]:
!pip install autogluon -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.6/880.6 kB 71.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 134.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 155.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 69.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
tobler 0.14.0 r

In [ ]:
#!/usr/bin/env python3
"""
Regression with AutoGluon on Moltbook data.
Uses embeddings, tabular features (including converted categoricals).
Compatible with NumPy 2.0.
"""
import os
from google.colab import drive
drive.mount('/content/drive')

# Install AutoGluon (and ensure torch is available)
!pip install autogluon -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from autogluon.tabular import TabularDataset, TabularPredictor

# ==========================================
# CONFIGURATION
# ==========================================
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'
DATA_PATH = f"{DESTINATION_DIR}/processed_v1_5_4_new_full.pkl"

EMBEDDING_COL = "embeddings"
TARGET_COL = "score"
RANDOM_STATE = 42
TEST_SIZE = 0.30
VAL_SIZE_FROM_TEMP = 0.50
DROP_COLS = ["safe_content", "content", "id"]

# AutoGluon settings
PRESETS = 'best_quality'   # Options: 'high_quality', 'good_quality', 'medium_quality'
TIME_LIMIT = 3600          # Time limit in seconds (None for no limit)
EVAL_METRIC = 'r2'         # Primary metric

# ==========================================
# 1. Load and prepare data
# ==========================================
print("\n" + "="*50)
print("LOADING AND PREPARING DATA")
print("="*50)

print("Loading data...")
moltbook = pd.read_pickle(DATA_PATH)
print(f"Original shape: {moltbook.shape}")

# Expand embeddings
embedding_lists = moltbook[EMBEDDING_COL].values
lengths = [len(lst) for lst in embedding_lists]
if len(set(lengths)) != 1:
    raise ValueError("Embedding lists have varying lengths.")
emb_dim = lengths[0]
print(f"Embedding dimension: {emb_dim}")

emb_df = pd.DataFrame(
    np.vstack(embedding_lists),
    index=moltbook.index,
    columns=[f"emb_{i}" for i in range(emb_dim)]
)

# Base features (all columns except target, embeddings, and dropped ones)
X_base = moltbook.drop(columns=[TARGET_COL, EMBEDDING_COL] + DROP_COLS)

# ==========================================
# CONVERT FORUM DUMMY COLUMNS TO CATEGORICAL
# ==========================================
print("\n" + "="*50)
print("CONVERTING FORUM COLUMNS TO CATEGORICAL")
print("="*50)

forum_columns = ['forum_philosophy', 'forum_technology', 'forum_todayilearned']
existing_forum_cols = [col for col in forum_columns if col in X_base.columns]
print(f"Found forum columns: {existing_forum_cols}")

if existing_forum_cols:
    # Verify one-hot encoding
    forum_sum = X_base[existing_forum_cols].sum(axis=1)
    if not (forum_sum <= 1).all():
        print("Warning: Forum columns are not mutually exclusive")
    
    # Create single categorical column
    X_base['forum'] = 'other'
    for col in existing_forum_cols:
        forum_name = col.replace('forum_', '')
        X_base.loc[X_base[col] == 1, 'forum'] = forum_name
    
    # Drop original dummies
    X_base = X_base.drop(columns=existing_forum_cols)
    print(f"Created 'forum' column with values: {X_base['forum'].unique().tolist()}")
    print(f"Forum value counts:\n{X_base['forum'].value_counts()}")
else:
    print("No forum columns found")

# ==========================================
# HANDLE HOUR AS CATEGORICAL
# ==========================================
print("\n" + "="*50)
print("CONVERTING HOUR TO CATEGORICAL")
print("="*50)

if 'hour' in X_base.columns:
    X_base['hour'] = X_base['hour'].astype(int)
    X_base['hour_category'] = X_base['hour'].astype(str)
    print(f"Hour values range: {X_base['hour'].min()} to {X_base['hour'].max()}")
    print(f"Created 'hour_category' with {X_base['hour_category'].nunique()} unique values")
    X_base = X_base.drop(columns=['hour'])
    print("Dropped original 'hour' column, using 'hour_category'")
else:
    print("No 'hour' column found")

# Ensure categorical columns are properly typed
for col in ['forum', 'hour_category']:
    if col in X_base.columns:
        X_base[col] = X_base[col].astype('category')

# ==========================================
# Target transformation
# ==========================================
y_raw = moltbook[TARGET_COL].clip(lower=0)
y = np.log1p(y_raw)   # log-transform
print(f"\nTarget range after log transform: [{y.min():.4f}, {y.max():.4f}]")

# ==========================================
# Train / Validation / Test Split
# ==========================================
print("\nSplitting data...")
X_base_train, X_base_temp, emb_train, emb_temp, y_train, y_temp = train_test_split(
    X_base, emb_df, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_base_val, X_base_test, emb_val, emb_test, y_val, y_test = train_test_split(
    X_base_temp, emb_temp, y_temp, test_size=VAL_SIZE_FROM_TEMP, random_state=RANDOM_STATE
)

print(f"Training set: {len(y_train)} samples")
print(f"Validation set: {len(y_val)} samples")
print(f"Test set: {len(y_test)} samples")

# ==========================================
# Combine into DataFrames for AutoGluon
# ==========================================
print("\n" + "="*50)
print("PREPARING DATAFRAMES FOR AUTOGLUON")
print("="*50)

# For AutoGluon we need one DataFrame per split containing all features + target
train_df = pd.concat([X_base_train.reset_index(drop=True),
                      emb_train.reset_index(drop=True),
                      y_train.reset_index(drop=True)], axis=1)
val_df   = pd.concat([X_base_val.reset_index(drop=True),
                      emb_val.reset_index(drop=True),
                      y_val.reset_index(drop=True)], axis=1)
test_df  = pd.concat([X_base_test.reset_index(drop=True),
                      emb_test.reset_index(drop=True),
                      y_test.reset_index(drop=True)], axis=1)

# Convert to AutoGluon's TabularDataset (optional but recommended)
train_data = TabularDataset(train_df)
val_data   = TabularDataset(val_df)
test_data  = TabularDataset(test_df)

print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

# ==========================================
# 3. Train AutoGluon Predictor
# ==========================================
print("\n" + "="*50)
print("TRAINING AUTOGLUON PREDICTOR")
print("="*50)
print(f"Presets: {PRESETS}")
print(f"Time limit: {TIME_LIMIT if TIME_LIMIT else 'None'} seconds")
print(f"Evaluation metric: {EVAL_METRIC}")

predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type='regression',
    eval_metric=EVAL_METRIC,
    path='/content/drive/MyDrive/autogluon_models'  # Save models to Drive
)

predictor.fit(
    train_data=train_data,
    tuning_data=val_data,       # Use validation set for tuning
    presets=PRESETS,
    time_limit=TIME_LIMIT,
    verbosity=2                 # Set to 3 for more detailed output
)

# ==========================================
# 4. Evaluate on test set
# ==========================================
print("\n" + "="*50)
print("EVALUATION ON TEST SET")
print("="*50)

# Leaderboard of all models
print("\nModel Leaderboard:")
leaderboard = predictor.leaderboard(test_data, silent=True)
print(leaderboard)

# Get predictions
test_preds = predictor.predict(test_data, model=predictor.model_best)
test_true = test_data[TARGET_COL].values

# Metrics in log space
test_r2 = r2_score(test_true, test_preds)
test_rmse = np.sqrt(mean_squared_error(test_true, test_preds))
test_mae = mean_absolute_error(test_true, test_preds)

print("\n" + "="*50)
print("TEST SET RESULTS (Log Space)")
print("="*50)
print(f"R² Score: {test_r2:.4f}")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE: {test_mae:.4f}")

# Convert back to original space
test_preds_original = np.expm1(test_preds)
test_true_original = np.expm1(test_true)
test_r2_original = r2_score(test_true_original, test_preds_original)
test_rmse_original = np.sqrt(mean_squared_error(test_true_original, test_preds_original))
test_mae_original = mean_absolute_error(test_true_original, test_preds_original)

print("\n" + "="*50)
print("TEST SET RESULTS (Original Space)")
print("="*50)
print(f"R² Score: {test_r2_original:.4f}")
print(f"RMSE: {test_rmse_original:.2f}")
print(f"MAE: {test_mae_original:.2f}")

# # ==========================================
# # 5. Feature Importance & Analysis
# # ==========================================
# print("\n" + "="*50)
# print("FEATURE IMPORTANCE")
# print("="*50)

# # Compute permutation importance on test set
# feature_importance = predictor.feature_importance(test_data, silent=True)
# print("\nTop 20 most important features:")
# print(feature_importance.head(20))

# # Plot feature importance
# fig, ax = plt.subplots(figsize=(10, 8))
# predictor.feature_importance(test_data).head(20).plot(kind='barh', ax=ax)
# ax.set_title('Top 20 Feature Importance')
# plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/autogluon_feature_importance.png', dpi=150, bbox_inches='tight')
# plt.show()

# # ==========================================
# # 6. Save additional results
# # ==========================================
# print("\n" + "="*50)
# print("SAVING ADDITIONAL RESULTS")
# print("="*50)

# results = {
#     'test_log_space': {
#         'r2': test_r2,
#         'rmse': test_rmse,
#         'mae': test_mae,
#     },
#     'test_original_space': {
#         'r2': test_r2_original,
#         'rmse': test_rmse_original,
#         'mae': test_mae_original,
#     },
#     'best_model': predictor.model_best,
#     'leaderboard': leaderboard.to_dict(),
#     'feature_importance': feature_importance.to_dict()
# }

# joblib.dump(results, '/content/drive/MyDrive/autogluon_results.pkl')
# print("Results saved to Google Drive")

# # ==========================================
# # 7. Prediction vs Actual Plot
# # ==========================================
# fig, ax = plt.subplots(figsize=(8, 6))
# ax.scatter(test_true, test_preds, alpha=0.5, s=2)
# ax.plot([test_true.min(), test_true.max()],
#         [test_true.min(), test_true.max()],
#         'r--', linewidth=2, label='Perfect Prediction')
# ax.set_xlabel('Actual (log space)')
# ax.set_ylabel('Predicted (log space)')
# ax.set_title(f'Predictions vs Actual (R² = {test_r2:.4f})')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('/content/drive/MyDrive/autogluon_predictions.png', dpi=150, bbox_inches='tight')
# plt.show()

# print("\n" + "="*50)
# print("AUTOGLUON TRAINING COMPLETE!")
# print("="*50)
# print(f"Best model: {predictor.model_best}")
# print(f"Test R² (log space): {test_r2:.4f}")
# print(f"Test R² (original space): {test_r2_original:.4f}")

Mounted at /content/drive


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject